# 01 — Coleta e Preparação dos Dados

**Objetivo:** executar o pipeline de extração e transformação das séries históricas, gerando o dataset analítico consolidado em `data/processed/dataset_analitico.csv`.

**Saídas:**
- `data/raw/` — séries brutas por fonte
- `data/processed/dataset_analitico.csv` — dataset mensal consolidado

---

## 0. Configuração do ambiente

In [ ]:
import sys
import os
from pathlib import Path

# adiciona src/ ao path para imports locais
sys.path.append(str(Path('..') / 'src'))

# carrega variáveis de ambiente (.env)
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path('..') / '.env')

import pandas as pd
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)

print('Ambiente configurado ✓')
print(f'FRED_API_KEY presente: {bool(os.getenv("FRED_API_KEY"))}')

## 1. Coleta das séries brutas

> ⏱ Esta célula faz chamadas a APIs externas. A execução pode levar 1–2 minutos.
> Se alguma série falhar, o erro será logado e as demais continuarão sendo coletadas.

In [ ]:
from extract import extract_all

series_brutas = extract_all()

print(f'\nSéries coletadas com sucesso: {len(series_brutas)}')
for nome, df in series_brutas.items():
    print(f'  {nome:<20} {len(df):>5} obs.  |  {df.index.min().date()} → {df.index.max().date()}')

### 1.1 Inspeção rápida — USD/BRL bruto

In [ ]:
df_cambio_raw = series_brutas.get('usd_brl')

if df_cambio_raw is not None:
    print(df_cambio_raw.head(10))
    print('...')
    print(df_cambio_raw.tail(5))
    print(f'\nShape: {df_cambio_raw.shape}')
    print(f'NaN: {df_cambio_raw.isna().sum().sum()}')

## 2. Transformação e consolidação

In [ ]:
from transform import run_transform_pipeline

dataset = run_transform_pipeline()

print(f'\nDataset consolidado: {dataset.shape[0]} meses × {dataset.shape[1]} colunas')
print(f'Período: {dataset.index.min().date()} → {dataset.index.max().date()}')

## 3. Verificação do dataset final

In [ ]:
dataset.head()

In [ ]:
dataset.tail()

In [ ]:
dataset.dtypes

### 3.1 Cobertura e valores ausentes

In [ ]:
from load import dataset_info
dataset_info()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# mapa visual de NaN
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    dataset.isna().T,
    cbar=False, cmap='Reds',
    xticklabels=False, yticklabels=True,
    ax=ax
)
ax.set_title('Mapa de Valores Ausentes por Variável', fontsize=12)
ax.set_xlabel('Tempo (meses)')
plt.tight_layout()
plt.show()

### 3.2 Estatísticas descritivas gerais

In [ ]:
dataset.describe().round(3)

## 4. Conclusão

O pipeline de coleta e preparação foi executado com sucesso. O dataset analítico está disponível em:

```
data/processed/dataset_analitico.csv
```

Próximo passo: **Notebook 02 — Análise Exploratória**.